# Sales Data Analysis and Prediction Using AI
**Beginner-level Data Analytics with AI project**

**Dataset:** UCI Machine Learning Repository — Online Retail  
**Dataset page:** https://archive.ics.uci.edu/dataset/352/online+retail  
**Author:** Satya Hari Kumari

## Project overview
This notebook explores historical online retail transactions, calculates sales metrics, visualizes trends, and builds a Random Forest regression model to estimate monthly revenue from calendar features and previous months' revenue.

> **Important:** This is historical data from 2010–2011, not current business data. The prediction is an educational demonstration, not a guarantee of future sales. Run all cells from top to bottom to generate the actual results and model metrics.

## 1. Objectives
- Load and understand a public retail transaction dataset.
- Clean missing values, duplicate rows, cancellations, and invalid quantities/prices.
- Calculate revenue as `Quantity × UnitPrice`.
- Explore monthly sales, leading products, and countries.
- Build and evaluate a simple machine-learning model for monthly revenue prediction.
- Explain findings and limitations in clear business language.

## 2. Install and import libraries
Install dependencies using `pip install -r requirements.txt` before running this notebook. The dataset is downloaded from the UCI direct link in the next section; an internet connection is required for the first run.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Libraries imported successfully.")

## 3. Load the dataset
The UCI Online Retail dataset is distributed as an Excel workbook. If automatic download is blocked by your network, download `Online Retail.xlsx` from the dataset page and place it in the same folder as this notebook, then set `LOCAL_FILE` to its filename.

In [ ]:
DATASET_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
LOCAL_FILE = "Online Retail.xlsx"

try:
    # Download and read the public dataset directly.
    df = pd.read_excel(DATASET_URL, engine="openpyxl")
    print("Dataset loaded from the UCI download link.")
except Exception as download_error:
    print("Direct download failed:", download_error)
    print("Trying a local copy:", LOCAL_FILE)
    df = pd.read_excel(LOCAL_FILE, engine="openpyxl")

print("Dataset shape:", df.shape)
display(df.head())

## 4. Understand the data
The dataset includes invoice number, product code and description, quantity, invoice date, unit price, customer ID, and country. A transaction's gross line value is calculated as quantity multiplied by unit price.

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nExact duplicate rows:", df.duplicated().sum())

## 5. Clean the data and calculate revenue
Data preparation choices:
- Remove exact duplicate rows.
- Exclude cancelled invoices (invoice numbers beginning with `C`).
- Keep only positive quantities and positive unit prices.
- Exclude rows with missing invoice date, product description, or country.
- Keep missing customer IDs because customer IDs are not needed for overall sales totals.

This creates a cleaned dataset of valid positive sales lines. These rules are appropriate for this demonstration, but different business questions may require different treatment of returns and cancellations.

In [ ]:
data = df.copy()
data = data.drop_duplicates()

data["InvoiceNo"] = data["InvoiceNo"].astype(str)
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"], errors="coerce")
data["Quantity"] = pd.to_numeric(data["Quantity"], errors="coerce")
data["UnitPrice"] = pd.to_numeric(data["UnitPrice"], errors="coerce")

data = data[
    ~data["InvoiceNo"].str.upper().str.startswith("C")
    & data["InvoiceDate"].notna()
    & data["Description"].notna()
    & data["Country"].notna()
    & data["Quantity"].gt(0)
    & data["UnitPrice"].gt(0)
].copy()

data["Revenue"] = data["Quantity"] * data["UnitPrice"]
data["Month"] = data["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

print("Original rows:", len(df))
print("Cleaned sales rows:", len(data))
print("Date range:", data["InvoiceDate"].min(), "to", data["InvoiceDate"].max())
print("Unique invoices:", data["InvoiceNo"].nunique())
print("Unique products:", data["StockCode"].nunique())
print("Countries:", data["Country"].nunique())
display(data.head())

## 6. Key performance indicators (KPIs)
Revenue is measured in the dataset's currency, British pounds (GBP). It is not profit because costs and expenses are not provided.

In [ ]:
total_revenue = data["Revenue"].sum()
total_orders = data["InvoiceNo"].nunique()
average_order_value = data.groupby("InvoiceNo")["Revenue"].sum().mean()
unique_products = data["StockCode"].nunique()
unique_customers = data["CustomerID"].nunique()

kpis = pd.DataFrame({
    "Metric": ["Total sales revenue (GBP)", "Unique orders", "Average order value (GBP)",
               "Unique products", "Unique customers with an ID"],
    "Value": [total_revenue, total_orders, average_order_value, unique_products, unique_customers]
})
display(kpis)

## 7. Monthly sales trend
A line chart helps identify changes in monthly revenue over the available historical period.

In [ ]:
monthly_sales = data.groupby("Month", as_index=False)["Revenue"].sum().sort_values("Month")

plt.figure(figsize=(12, 5))
plt.plot(monthly_sales["Month"], monthly_sales["Revenue"], marker="o")
plt.title("Monthly Sales Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (GBP)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(monthly_sales)

## 8. Top products by revenue
Product descriptions can be inconsistent or missing in raw transactions; this chart uses the available cleaned descriptions.

In [ ]:
top_products = (
    data.groupby("Description", as_index=False)["Revenue"]
    .sum()
    .sort_values("Revenue", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(top_products["Description"].astype(str)[::-1], top_products["Revenue"][::-1])
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (GBP)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

display(top_products)

## 9. Sales by country
The chart shows the ten countries with the highest recorded revenue in this dataset.

In [ ]:
country_sales = (
    data.groupby("Country", as_index=False)["Revenue"]
    .sum()
    .sort_values("Revenue", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(country_sales["Country"][::-1], country_sales["Revenue"][::-1])
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue (GBP)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

display(country_sales)

## 10. Prepare monthly data for AI prediction
For a beginner-friendly forecast, the model predicts a month's revenue using calendar features and the previous month's revenue. The first month has no previous month in the dataset, so it is removed from model training.

The split is chronological: earlier months are used for training and the latest months for testing. This is more appropriate for time-ordered data than randomly mixing months.

In [ ]:
forecast_data = monthly_sales.copy().sort_values("Month").reset_index(drop=True)
forecast_data["MonthNumber"] = forecast_data["Month"].dt.month
forecast_data["Year"] = forecast_data["Month"].dt.year
forecast_data["PreviousMonthRevenue"] = forecast_data["Revenue"].shift(1)
forecast_data = forecast_data.dropna().reset_index(drop=True)

feature_columns = ["MonthNumber", "Year", "PreviousMonthRevenue"]
target_column = "Revenue"

if len(forecast_data) < 6:
    raise ValueError(
        "Not enough monthly observations to create a meaningful train/test split. "
        "Check that the dataset was loaded correctly."
    )

split_index = max(1, int(len(forecast_data) * 0.8))
split_index = min(split_index, len(forecast_data) - 1)

X = forecast_data[feature_columns]
y = forecast_data[target_column]

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Training months:", len(X_train))
print("Testing months:", len(X_test))
print("Training period:", forecast_data["Month"].iloc[0], "to", forecast_data["Month"].iloc[split_index - 1])
print("Testing period:", forecast_data["Month"].iloc[split_index], "to", forecast_data["Month"].iloc[-1])

## 11. Train a Random Forest regression model
Random Forest combines predictions from multiple decision trees. It can learn non-linear patterns, but a small monthly dataset means results should be treated cautiously.

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    min_samples_leaf=1
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame({
    "Metric": ["Mean Absolute Error (GBP)", "Root Mean Squared Error (GBP)", "R-squared"],
    "Value": [mae, rmse, r2]
})
display(metrics)

### How to interpret the metrics
- **MAE:** average absolute difference between actual and predicted revenue; lower is better.
- **RMSE:** similar to MAE but penalizes large errors more strongly; lower is better.
- **R-squared:** compares model performance with a simple average-based baseline; it can be negative on a test set and is unstable with very few test months.

Always interpret the metrics alongside the number of test months. A good score on this short historical series does not prove the model will predict future business sales accurately.

## 12. Compare actual and predicted revenue

In [ ]:
comparison = pd.DataFrame({
    "Month": forecast_data["Month"].iloc[split_index:].to_numpy(),
    "ActualRevenue": y_test.to_numpy(),
    "PredictedRevenue": predictions
})
display(comparison)

plt.figure(figsize=(11, 5))
plt.plot(comparison["Month"], comparison["ActualRevenue"], marker="o", label="Actual")
plt.plot(comparison["Month"], comparison["PredictedRevenue"], marker="o", label="Predicted")
plt.title("Actual vs Predicted Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (GBP)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Feature importance
Feature importance gives a rough indication of which model inputs were most useful to the fitted Random Forest. It does not establish causation.

In [ ]:
importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

plt.figure(figsize=(8, 4))
plt.barh(importance["Feature"][::-1], importance["Importance"][::-1])
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 14. Export useful results
The following files are saved to the current working directory and can be included with the project if required.

In [ ]:
monthly_sales.to_csv("monthly_sales_summary.csv", index=False)
top_products.to_csv("top_products_by_revenue.csv", index=False)
country_sales.to_csv("country_sales_summary.csv", index=False)
comparison.to_csv("actual_vs_predicted_monthly_sales.csv", index=False)

print("Saved: monthly_sales_summary.csv")
print("Saved: top_products_by_revenue.csv")
print("Saved: country_sales_summary.csv")
print("Saved: actual_vs_predicted_monthly_sales.csv")

## 15. Conclusions and recommendations
Use the outputs above to write evidence-based conclusions. For example:
- Identify the months with the highest and lowest recorded revenue.
- Identify products and countries that contribute the most revenue.
- Describe model error using MAE and RMSE, and state the number of test months.
- Recommend that a retailer monitor seasonality, product demand, and country-level sales.

Do not claim that sales increased or decreased unless the displayed results support that statement. Revenue is not profit, and the dataset is historical.

## 16. Dataset citation
Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

Dataset page: https://archive.ics.uci.edu/dataset/352/online+retail

The dataset is licensed under Creative Commons Attribution 4.0 (CC BY 4.0). Give attribution when sharing adapted work.